# Feature Selection

### Setup

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# Setup working directory
from pathlib import Path
import os

def find_root_dir(marker='tfg'):
    p = Path.cwd()
    for candidate in [p] + list(p.parents):
        if candidate.name == marker:
            return candidate.resolve()
        if (candidate / marker).is_dir():
            return (candidate / marker).resolve()
    raise FileNotFoundError(f"Could not find '{marker}' folder in {Path.cwd()} or its parents.")

os.chdir(find_root_dir('tfg'))

## Load Data

In [3]:
ENCODED_DATASET_PATH = "data/processed/stage1/reference-8k-encoded.parquet"

In [4]:
from src.preprocessing import load_dataset
df = load_dataset(ENCODED_DATASET_PATH)
print("\nFirst few rows:")
df.head()


First few rows:


,Source IP,Destination IP,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,...,Label,is_0,is_6,is_17,Source Port_Dynamic,Source Port_Registered,Source Port_Well-known,Destination Port_Dynamic,Destination Port_Registered,Destination Port_Well-known
0,0.770547,0.87135,1148952,5,0,30,0,6,6,6.000000,...,DDoS,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,0.770547,0.87135,8361212,4,0,24,0,6,6,6.000000,...,DDoS,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.770547,0.87135,1280276,2,5,20,11607,20,0,10.000000,...,DDoS,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,0.770547,0.87135,78218,3,6,26,11601,20,0,8.666667,...,DDoS,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4,0.865553,0.87135,223,2,2,60,414,30,30,30.000000,...,BENIGN,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0


## Data Partitioning
- Split data to avoid **data leakage**
- Not necessary to encode target for feature selection
- Keep same ``random_state`` value for **training**

In [5]:
from src.preprocessing import split_features_target
from src.train import split_data

TARGET_COLUMN = "Label"
SEED = 42
X, y = split_features_target(df, TARGET_COLUMN)
X_train, y_train, X_test, y_test = split_data(X,y,test_size=0.2, random_state=SEED)

## Correlation Analysis
- Determine the selected feature names using ``X_train``, then apply feature selection to the complete dataset loaded from ``stage1/``. This avoids saving separate train/test files.

In [6]:
import numpy as np
import pandas as pd

def findCorrelation(corr, cutoff=0.9, exact=None):
    def _findCorrelation_fast(corr, avg, cutoff):
        combsAboveCutoff = corr.where(lambda x: (np.tril(x)==0) & (x > cutoff)).stack().index
        rowsToCheck = combsAboveCutoff.get_level_values(0)
        colsToCheck = combsAboveCutoff.get_level_values(1)
        msk = avg[colsToCheck] > avg[rowsToCheck].values
        deletecol = pd.unique(np.r_[colsToCheck[msk], rowsToCheck[~msk]]).tolist()
        return deletecol
    def _findCorrelation_exact(corr, avg, cutoff):
        x = corr.loc[(*[avg.sort_values(ascending=False).index]*2,)]
        if (x.dtypes.values[:, None] == ['int64', 'int32', 'int16', 'int8']).any():
            x = x.astype(float)
        x.values[(*[np.arange(len(x))]*2,)] = np.nan
        deletecol = []
        for ix, i in enumerate(x.columns[:-1]):
            for j in x.columns[ix+1:]:
                if x.loc[i, j] > cutoff:
                    if x[i].mean() > x[j].mean():
                        deletecol.append(i)
                        x.loc[i] = x[i] = np.nan
                    else:
                        deletecol.append(j)
                        x.loc[j] = x[j] = np.nan
        return deletecol
    
    if not np.allclose(corr, corr.T) or any(corr.columns!=corr.index):
        raise ValueError("correlation matrix is not symmetric.")
    acorr = corr.abs()
    avg = acorr.mean()
    if exact or exact is None and corr.shape[1]<100:
        return _findCorrelation_exact(acorr, avg, cutoff)
    else:
        return _findCorrelation_fast(acorr, avg, cutoff)


In [7]:
from src.preprocessing import save
from src.utils import intro

corr = X_train.corr()
feature_sets = {}

for threshold in [0.8, 0.9]:
    hc = findCorrelation(corr, cutoff=threshold)

    selected_features = [
        column
        for column in X_train.columns
        if column not in hc
    ]

    feature_set_name = f"correlation_{str(threshold).replace('.', '')}"
    feature_sets[feature_set_name] = selected_features

    intro(f"Threshold {threshold:.2f}: {len(hc)} features removed, {len(selected_features)} features remaining")

    df_selected = df.drop(columns=hc)
    save(f"stage1/{feature_set_name}.parquet", df_selected)


Threshold 0.80: 43 features removed, 35 features remaining
Saving dataset as .parquet
Final Dataset Saved: data/processed\stage1/correlation_08.parquet (0.37 MB) 
Threshold 0.90: 37 features removed, 41 features remaining
Saving dataset as .parquet
Final Dataset Saved: data/processed\stage1/correlation_09.parquet (0.47 MB) 
